In [1]:
# ─── CELL 1: Install / upgrade dependencies ────────────────

# !pip install -q transformers==4.40.0 datasets accelerate evaluate scikit-learn tqdm

In [2]:
# ─── CELL 2: Imports ───────────────────────────────────────

import os, gc, math, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    average_precision_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

In [3]:
# ─── CELL 3: Reproducibility & device setup ────────────────

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

NUM_GPUS = torch.cuda.device_count()
print(f"GPUs available: {NUM_GPUS}")
for i in range(NUM_GPUS):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Primary device: {DEVICE}")

GPUs available: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
Primary device: cuda


In [4]:
# ─── CELL 4: Configuration ─────────────────────────────────

MODEL_NAME = "TechWolf/JobBERT-v3"
OUTPUT_DIR = "/kaggle/working/jobbertv3_output"
LOGGING_DIR = "/kaggle/working/jobbertv3_logs"

# Paths
TRAIN_CSV = (
    "/kaggle/input/datasets/danielantoniudumitru/"
    "talentclef-subtaska/Training/normalization/"
    "normalized_job_applicant_dataset.csv"
)
DEV_BASE = "/kaggle/input/datasets/danielantoniudumitru/talentclef-subtaska/Development/en"
CORPUS_DIR = os.path.join(DEV_BASE, "corpus")
QUERY_DIR = os.path.join(DEV_BASE, "queries")
QRELS_FILE = os.path.join(DEV_BASE, "qrels.tsv")

# Hyper-parameters
MAX_LEN = 512
TRAIN_BATCH = 8
EVAL_BATCH = 16
GRAD_ACCUM = 4
NUM_EPOCHS = 4
LR = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

os.makedirs(OUTPUT_DIR,  exist_ok=True)
os.makedirs(LOGGING_DIR, exist_ok=True)

In [5]:
# ─── CELL 5: Load & inspect training data ──────────────────

# Currently removed to make the whole model only on the development set

# print("Loading training CSV …")
# df_train = pd.read_csv(TRAIN_CSV)
# print(f"Shape : {df_train.shape}")
# print(df_train.head(3))
# print("\nColumn dtypes:\n", df_train.dtypes)
# print("\nValue counts (label):\n", df_train.iloc[:, -1].value_counts())

In [6]:
# ─── CELL 6: Define columns (hardcoded for this dataset) ───

text_a_col = "query_text"
text_b_col = "corpus_text"
label_col = "relevance"

print(f"Text A: {text_a_col} | Text B: {text_b_col} | Label: {label_col}")

Text A: query_text | Text B: corpus_text | Label: relevance


In [7]:
# ─── CELL 7: Train / validation split ──────────────────────

# Currently removed to make the split on the development set

# df_train_split, df_val_split = train_test_split(
#     df_train, test_size=0.1, stratify=df_train[label_col], random_state=SEED
# )
# print(f"Train size : {len(df_train_split)}")
# print(f"Val size : {len(df_val_split)}")

In [8]:
# ─── CELL 8: Tokenizer ─────────────────────────────────────

print(f"\nLoading tokenizer: {MODEL_NAME} …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


Loading tokenizer: TechWolf/JobBERT-v3 …


config.json:   0%|          | 0.00/697 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

In [9]:
# ─── CELL 9: Helper — read dev corpus / queries ───────────

def read_text_file(folder, file_id):
    path = os.path.join(folder, str(file_id))
    if not os.path.exists(path):
        return ""
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return f.read().strip()

In [10]:
# ─── CELL 10: Load qrels ───────────────────────────────────

print("\nLoading qrels …")
qrels = pd.read_csv(
    QRELS_FILE, sep="\t", header=None,
    names=["q_id", "iter", "c_id", "relevance"]
)
print(f"qrels shape: {qrels.shape}")
print(qrels.head())


Loading qrels …
qrels shape: (472, 4)
    q_id  iter   c_id  relevance
0  36044     0  13884          1
1  39060     0   9516          1
2  39060     0  12097          1
3  32447     0  13882          1
4  39060     0   6533          1


In [11]:
# ─── CELL 11: Build dev pairs ───────────────────────────────

print("\nBuilding (query, corpus) pairs …")
all_q_ids = qrels["q_id"].unique().tolist()
all_c_ids = qrels["c_id"].unique().tolist()
positive_set = set(zip(qrels["q_id"], qrels["c_id"]))

dev_records = []

for _, row in tqdm(qrels.iterrows(), total=len(qrels), desc="Reading positive pairs"):
    q_text = read_text_file(QUERY_DIR,  row["q_id"])
    c_text = read_text_file(CORPUS_DIR, row["c_id"])
    dev_records.append({
        "q_id": row["q_id"],
        "c_id": row["c_id"],
        "query_text": q_text,
        "corpus_text": c_text,
        "relevance": 1
    })

neg_pairs = [
    (q, c)
    for q in all_q_ids
    for c in all_c_ids
    if (q, c) not in positive_set
]

for q_id, c_id in tqdm(neg_pairs, desc="Reading negative pairs"):
    q_text = read_text_file(QUERY_DIR,  q_id)
    c_text = read_text_file(CORPUS_DIR, c_id)
    dev_records.append({
        "q_id": q_id,
        "c_id": c_id,
        "query_text": q_text,
        "corpus_text": c_text,
        "relevance": 0
    })

df_all = pd.DataFrame(dev_records)
print(f"Total pairs: {len(df_all)}")
print(f"Positives: {(df_all['relevance']==1).sum()}")
print(f"Negatives: {(df_all['relevance']==0).sum()}")

df_train_split, df_temp = train_test_split(
    df_all, test_size=0.30, random_state=SEED, stratify=df_all["relevance"]
)
df_val_split, df_eval_split = train_test_split(
    df_temp, test_size=(2/3), random_state=SEED, stratify=df_temp["relevance"]
)

print(f"Train: {len(df_train_split)}")
print(f"Val: {len(df_val_split)}")
print(f"Eval: {len(df_eval_split)}")


Building (query, corpus) pairs …


Reading positive pairs:   0%|          | 0/472 [00:00<?, ?it/s]

Reading negative pairs:   0%|          | 0/3728 [00:00<?, ?it/s]

Total pairs: 4200
Positives: 472
Negatives: 3728
Train: 2940
Val: 420
Eval: 840


In [12]:
# ─── CELL 12: Dataset class ─────────────────────────────────

class PairDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len,
                 col_a, col_b, label_col):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.col_a = col_a
        self.col_b = col_b
        self.label_col = label_col

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row  = self.data.iloc[idx]

        def clean_text(t):
            return " ".join(str(t).split()[:256]) if pd.notna(t) else ""

        text_a = clean_text(row[self.col_a]) if pd.notna(row[self.col_a]) else ""
        text_b = clean_text(row[self.col_b]) if pd.notna(row[self.col_b]) else ""
        
        encoding = self.tokenizer(
            text_a,
            text_b,
            max_length=self.max_len,
            truncation=True,
            padding=False,
            return_token_type_ids=True
        )
        encoding["labels"] = int(row[self.label_col])
        return encoding


train_dataset = PairDataset(df_train_split, tokenizer, MAX_LEN,
                             text_a_col, text_b_col, label_col)
val_dataset = PairDataset(df_val_split,  tokenizer, MAX_LEN,
                             text_a_col, text_b_col, label_col)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"Sample encoding keys: {list(train_dataset[0].keys())}")

Sample encoding keys: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']


In [13]:
# ─── CELL 13: Model ────────────────────────────────────────

print(f"\nLoading model: {MODEL_NAME} …")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")


Loading model: TechWolf/JobBERT-v3 …


model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: TechWolf/JobBERT-v3
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model parameters: 278,045,186


In [14]:
# ─── CELL 14: Metrics ──────────────────────────────────────

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    preds = (probs >= 0.5).astype(int)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, zero_division=0),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "ap": average_precision_score(labels, probs)
    }

In [15]:
# ─── CELL 15: TrainingArguments ────────────────────────────

steps_per_epoch = math.ceil(len(train_dataset) / (TRAIN_BATCH * max(NUM_GPUS, 1) * GRAD_ACCUM))
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

os.environ["TENSORBOARD_LOGGING_DIR"] = LOGGING_DIR

training_args = TrainingArguments(
    output_dir = OUTPUT_DIR,
    logging_dir = LOGGING_DIR,

    # Epochs & batch
    num_train_epochs = NUM_EPOCHS,
    per_device_train_batch_size = TRAIN_BATCH,
    per_device_eval_batch_size = EVAL_BATCH,
    gradient_accumulation_steps = GRAD_ACCUM,

    # Optimiser
    learning_rate = LR,
    weight_decay = WEIGHT_DECAY,
    warmup_steps = warmup_steps,
    lr_scheduler_type = "cosine",

    # Evaluation & saving
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    metric_for_best_model = "ap",
    greater_is_better = True,

    # Logging
    report_to = "none",

    # Multi-GPU / precision
    fp16 = True,
    dataloader_num_workers = 4,
    seed = SEED,

    # Progress bar
    disable_tqdm = False
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [16]:
# ─── CELL 16: Trainer ──────────────────────────────────────

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    processing_class = tokenizer,
    data_collator = data_collator,
    compute_metrics = compute_metrics,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=2)]
)

In [17]:
# ─── CELL 17: Training ─────────────────────────────────────

print("\n" + "="*60)
print("  Starting JobBERT fine-tuning …")
print("="*60)

train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("\nTraining summary:")
print(train_result.metrics)


  Starting JobBERT fine-tuning …


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Ap
1,No log,0.544368,0.907143,0.417910,0.700000,0.297872,0.561355
2,No log,0.452655,0.916667,0.606742,0.642857,0.574468,0.662716
3,No log,0.330362,0.940476,0.698795,0.805556,0.617021,0.826964
4,No log,0.238519,0.952381,0.743590,0.935484,0.617021,0.897962


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training summary:
{'train_runtime': 816.2734, 'train_samples_per_second': 14.407, 'train_steps_per_second': 0.225, 'total_flos': 3094186011033600.0, 'train_loss': 1.7503934113875679, 'epoch': 4.0}


In [18]:
# ─── CELL 18: Dev Dataset class ────────────────────────────

class DevPairDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        def clean_text(t):
            return " ".join(str(t).split()[:256]) if pd.notna(t) else ""

        row = self.data.iloc[idx]
        enc = self.tokenizer(
            clean_text(row["query_text"]),
            clean_text(row["corpus_text"]),
            max_length=self.max_len,
            truncation=True,
            padding=False,
            return_token_type_ids=True,
        )
        enc["labels"] = int(row["relevance"])
        return enc

dev_dataset = DevPairDataset(df_eval_split, tokenizer, MAX_LEN)

In [19]:
# ─── CELL 19: Inference on dev set ─────────────────────────

print("\nRunning predictions on dev set …")

raw_preds = trainer.predict(dev_dataset)
logits = raw_preds.predictions
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
preds_bin = (probs >= 0.5).astype(int)
labels = raw_preds.label_ids

df_eval_split["pred_prob"] = probs
df_eval_split["pred_label"] = preds_bin


Running predictions on dev set …


In [20]:
# ─── CELL 20: Ranking Metrics ──────────────────────────────

def mean_average_precision(df, q_col="q_id", prob_col="pred_prob", rel_col="relevance"):
    aps = []
    for qid, grp in df.groupby(q_col):
        grp_sorted = grp.sort_values(prob_col, ascending=False).reset_index(drop=True)
        rel = grp_sorted[rel_col].values
        if rel.sum() == 0:
            continue
        hits, ap = 0, 0.0
        for rank, r in enumerate(rel, start=1):
            if r == 1:
                hits += 1
                ap += hits / rank
        aps.append(ap / rel.sum())
    return np.mean(aps) if aps else 0.0


def mean_reciprocal_rank(df, q_col="q_id", prob_col="pred_prob", rel_col="relevance"):
    rrs = []
    for qid, grp in df.groupby(q_col):
        grp_sorted = grp.sort_values(prob_col, ascending=False).reset_index(drop=True)
        rel = grp_sorted[rel_col].values
        rr = 0.0
        for rank, r in enumerate(rel, start=1):
            if r == 1:
                rr = 1.0 / rank
                break
        rrs.append(rr)
    return np.mean(rrs) if rrs else 0.0


def precision_at_k(df, k, q_col="q_id", prob_col="pred_prob", rel_col="relevance"):
    pk_list = []
    for qid, grp in df.groupby(q_col):
        grp_sorted = grp.sort_values(prob_col, ascending=False).head(k)
        pk_list.append(grp_sorted[rel_col].mean())
    return np.mean(pk_list) if pk_list else 0.0

In [21]:
# ─── CELL 21: Print all metrics ────────────────────────────

acc = accuracy_score(labels, preds_bin)
f1 = f1_score(labels, preds_bin, zero_division=0)
prec = precision_score(labels, preds_bin, zero_division=0)
rec = recall_score(labels, preds_bin, zero_division=0)
ap_overall = average_precision_score(labels, probs)

MAP = mean_average_precision(df_eval_split)
MRR = mean_reciprocal_rank(df_eval_split)
P_at1 = precision_at_k(df_eval_split, k=1)
P_at5 = precision_at_k(df_eval_split, k=5)
P_at10 = precision_at_k(df_eval_split, k=10)

print("\n" + "="*60)
print("  JobBERTv3 — Dev Set Evaluation Results")
print("="*60)
print(f"  Accuracy: {acc:.4f}")
print(f"  F1: {f1:.4f}")
print(f"  Precision (binary): {prec:.4f}")
print(f"  Recall: {rec:.4f}")
print(f"  Average Precision: {ap_overall:.4f}")
print("─"*60)
print(f"  MAP(Mean Avg Precision): {MAP:.4f}")
print(f"  MRR(Mean Recip Rank): {MRR:.4f}")
print(f"  P@1: {P_at1:.4f}")
print(f"  P@5: {P_at5:.4f}")
print(f"  P@10: {P_at10:.4f}")
print("="*60)


  JobBERTv3 — Dev Set Evaluation Results
  Accuracy: 0.9548
  F1: 0.7791
  Precision (binary): 0.8701
  Recall: 0.7053
  Average Precision: 0.8971
────────────────────────────────────────────────────────────
  MAP(Mean Avg Precision): 0.9010
  MRR(Mean Recip Rank): 1.0000
  P@1: 1.0000
  P@5: 0.8400
  P@10: 0.6400


In [22]:
# ─── CELL 22: Save predictions ─────────────────────────────

df_eval_split.to_csv("/kaggle/working/jobbert_dev_predictions.csv", index=False)
print("\nPredictions saved to /kaggle/working/jobbert_dev_predictions.csv")


Predictions saved to /kaggle/working/jobbert_dev_predictions.csv
